<a href="https://colab.research.google.com/github/EgzonnOsmanaj/MesoAI/blob/main/Medical_data_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install faiss-cpu sentence-transformers scikit-learn pandas numpy tqdm transformers torch

In [ ]:
import os
import re
import json
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import faiss
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

random.seed(42)
np.random.seed(42)

In [ ]:
DATA_URL = "https://www.dropbox.com/scl/fi/54p9kkx5n93bffyx08eba/textbooks.zip?rlkey=2y2c5x8y0uncnddichn9cmd7n&st=m290nmkk&dl=1"
DATA_DIR = Path("data_textbooks")
DATA_DIR.mkdir(exist_ok=True)

zip_path = DATA_DIR / "textbooks.zip"

if not zip_path.exists():
    import requests
    r = requests.get(DATA_URL, stream=True, timeout=60)
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(DATA_DIR)

text_dir = DATA_DIR / "textbooks" / "en"
text_files = sorted(text_dir.glob("*.txt"))

len(text_files)

In [ ]:
def load_docs(files):
    docs = []
    for fp in files:
        txt = fp.read_text(encoding="utf-8", errors="ignore")
        docs.append({"source": fp.name, "text": txt})
    return docs

docs = load_docs(text_files)
len(docs), docs[0]["source"]

In [ ]:
def clean_text(t):
    t = re.sub(r"\s+", " ", t)
    return t.strip()

def chunk_words(text, size=220, overlap=40):
    words = text.split()
    out = []
    i = 0
    while i < len(words):
        out.append(" ".join(words[i:i+size]))
        i += max(1, size - overlap)
    return out

In [ ]:
chunks = []
for d in docs:
    clean = clean_text(d["text"])
    for idx, ch in enumerate(chunk_words(clean)):
        chunks.append({
            "doc": d["source"],
            "chunk_id": idx,
            "text": ch
        })

chunks_df = pd.DataFrame(chunks)
chunks_df.head()

In [ ]:
out_dir = Path("output")
out_dir.mkdir(exist_ok=True)

chunks_df.to_csv(out_dir / "chunked_documents.csv", index=False)
len(chunks_df)

In [ ]:
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

embeddings = embed_model.encode(
    chunks_df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype("float32"))

vectorizer = TfidfVectorizer(stop_words="english", max_features=50000)
tfidf = vectorizer.fit_transform(chunks_df["text"].tolist())

In [ ]:
def baseline_retrieve(query, top_k=8):
    q = embed_model.encode([query], normalize_embeddings=True)
    scores, idx = index.search(q.astype("float32"), top_k)
    return chunks_df.iloc[idx[0]].assign(score=scores[0]).reset_index(drop=True)

In [ ]:
def lexical_retrieve(query, top_k=8):
    q = vectorizer.transform([query])
    scores = (tfidf @ q.T).toarray().ravel()
    idx = np.argsort(-scores)[:top_k]
    return chunks_df.iloc[idx].assign(score=scores[idx]).reset_index(drop=True)

def hybrid_retrieve(query, top_k=8, pool_k=20, alpha=0.6):
    dense = baseline_retrieve(query, pool_k).copy()
    lex = lexical_retrieve(query, pool_k).copy()

    dense["dense_score"] = dense["score"]
    lex["lex_score"] = lex["score"]

    pool = pd.concat(
        [dense[["doc", "chunk_id", "text", "dense_score"]],
         lex[["doc", "chunk_id", "text", "lex_score"]]],
        ignore_index=True
    )

    pool = pool.groupby(["doc", "chunk_id", "text"], as_index=False).max(numeric_only=True).fillna(0)

    pairs = [(query, t) for t in pool["text"].tolist()]
    pool["rerank_score"] = reranker.predict(pairs)

    pool["combined_score"] = (
        alpha * pool.get("dense_score", 0) +
        (1 - alpha) * pool.get("lex_score", 0)
    )

    return pool.sort_values("rerank_score", ascending=False).head(top_k).reset_index(drop=True)

In [ ]:
gen_model_name = "google/flan-t5-base"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(gen_model_name)

In [ ]:
def build_context(rows):
    return "\n\n".join([f"[{i+1}] {r.text}" for i, r in rows.iterrows()])

def answer(query, rows):
    context = build_context(rows)
    prompt = (
        "Answer the question using only the context. "
        "If the context is insufficient, say you cannot determine it.\n\n"
        f"Question: {query}\n\n"
        f"Context:\n{context}\n\n"
        "Answer:"
    )
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    out = gen_model.generate(**inputs, max_new_tokens=180)
    return gen_tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
test_set = [
    {"query": "What is the definition of heart failure?", "answer_key": "The heart cannot pump enough blood to meet tissue demands."},
    {"query": "What are common causes of systolic dysfunction?", "answer_key": "Ischemic heart disease, hypertension, cardiomyopathy, infarction."},
    {"query": "How does diastolic dysfunction differ from systolic dysfunction?", "answer_key": "Relaxation and filling problem rather than contraction problem."},
    {"query": "What causes pulmonary edema in left-sided heart failure?", "answer_key": "Pulmonary congestion from elevated left-sided pressures."},
    {"query": "Name one cause of right-sided heart failure besides left-sided failure.", "answer_key": "Cor pulmonale or left-to-right shunt."},
    {"query": "Which mechanism can lead to sudden cardiac death in athletes?", "answer_key": "Hypertrophic cardiomyopathy."},
    {"query": "What are the causes of heart failure?", "answer_key": "Multiple mechanisms including systolic/diastolic dysfunction, valve disease, obstruction, shunts, arrhythmias, rupture, and high-output states."},
    {"query": "What happens if the context does not contain the answer?", "answer_key": "The model should state insufficient information."},
]

In [ ]:
def retrieval_hit(rows, key_terms):
    text = " ".join(rows["text"].tolist()).lower()
    return int(any(term.lower() in text for term in key_terms))

retrieval_results = []
for item in test_set:
    q = item["query"]
    base = baseline_retrieve(q, 5)
    enh = hybrid_retrieve(q, 5)

    key_terms = item["answer_key"].split()[:4]
    retrieval_results.append({
        "query": q,
        "baseline_hit": retrieval_hit(base, key_terms),
        "enhanced_hit": retrieval_hit(enh, key_terms)
    })

retrieval_df = pd.DataFrame(retrieval_results)
retrieval_df

In [ ]:
def score_answer(ans, key):
    a = ans.lower()
    k = key.lower()
    correct = int(any(tok in a for tok in k.split()[:5]))
    grounded = int("cannot determine" in a or len(a.split()) > 10)
    complete = int(len(a.split()) > 20)
    return correct, grounded, complete

gen_rows = []
for item in test_set:
    q = item["query"]

    base_rows = baseline_retrieve(q, 5)
    enh_rows = hybrid_retrieve(q, 5)

    base_ans = answer(q, base_rows)
    enh_ans = answer(q, enh_rows)

    b = score_answer(base_ans, item["answer_key"])
    e = score_answer(enh_ans, item["answer_key"])

    gen_rows.append({
        "query": q,
        "baseline_correct": b[0],
        "baseline_grounded": b[1],
        "baseline_complete": b[2],
        "enhanced_correct": e[0],
        "enhanced_grounded": e[1],
        "enhanced_complete": e[2]
    })

gen_df = pd.DataFrame(gen_rows)
gen_df

In [ ]:
summary = pd.DataFrame([
    {
        "setting": "baseline",
        "retrieval_hit_rate": retrieval_df["baseline_hit"].mean(),
        "correct_rate": gen_df["baseline_correct"].mean(),
        "grounded_rate": gen_df["baseline_grounded"].mean(),
        "complete_rate": gen_df["baseline_complete"].mean()
    },
    {
        "setting": "enhanced",
        "retrieval_hit_rate": retrieval_df["enhanced_hit"].mean(),
        "correct_rate": gen_df["enhanced_correct"].mean(),
        "grounded_rate": gen_df["enhanced_grounded"].mean(),
        "complete_rate": gen_df["enhanced_complete"].mean()
    }
])

summary

In [ ]:
retrieval_df.to_csv(out_dir / "retrieval_evaluation.csv", index=False)
gen_df.to_csv(out_dir / "generation_evaluation.csv", index=False)
summary.to_csv(out_dir / "summary_metrics.csv", index=False)
summary

In [ ]:
query = "What are the causes of heart failure?"
base_ctx = baseline_retrieve(query, 5)
enh_ctx = hybrid_retrieve(query, 5)

print("BASELINE ANSWER:")
print(answer(query, base_ctx))
print("\nENHANCED ANSWER:")
print(answer(query, enh_ctx))